# SafeRest360 — Pipeline de Entrenamiento

Este notebook ejecuta el pipeline completo de Transfer Learning para detección de personas:
1. Preparación del dataset (COCO 2017 → clase persona → formato YOLO)
2. Entrenamiento de 4 variantes YOLO
3. Evaluación y comparación

**Requisitos**: GPU runtime (T4 o superior)

In [ ]:
# Verificar GPU
!nvidia-smi
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
# Instalar dependencias (supervision 0.31 eliminó ByteTrack)
!pip install -q ultralytics albumentations "supervision<0.31" mlflow
# Dependencias de compilación para YOLO-NAS
!apt-get install -qq python3-dev > /dev/null 2>&1
!pip install -q pycocotools
# YOLO-NAS (Apache-2.0) — no crítico para entrenar YOLOv8
!pip install -q super-gradients || echo "YOLO-NAS install failed (non-critical for YOLOv8 training)"

In [ ]:
# Clonar repositorio desde GitHub (si no existe)
import os
if not os.path.exists('SafeRest360'):
    !git clone https://github.com/maicolchavarro/SafeRest360.git
else:
    !git -C SafeRest360 pull

import sys
if 'SafeRest360/vision-engine' not in sys.path:
    sys.path.insert(0, 'SafeRest360/vision-engine')

## 1. Descarga y preparación del dataset COCO 2017

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Descargar anotaciones COCO 2017 (si no existen)
if not os.path.exists('data/annotations/instances_val2017.json'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O data/annotations.zip
    !unzip -qo data/annotations.zip -d data/
else:
    print("Anotaciones ya descargadas")

# Descargar imágenes val2017 (si no existen)
if not os.path.exists('data/val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O data/val2017.zip
    !unzip -qo data/val2017.zip -d data/
else:
    print("Imágenes val2017 ya descargadas")

# Para entrenamiento completo, descomentar:
# !wget -q http://images.cocodataset.org/zips/train2017.zip -O data/train2017.zip
# !unzip -qo data/train2017.zip -d data/

In [ ]:
from pathlib import Path
from src.dataset.coco_extractor import extract_person_subset, create_dataset_yaml

# Extraer solo personas de COCO val2017
stats = extract_person_subset(
    annotations_path=Path('data/annotations/instances_val2017.json'),
    images_dir=Path('data/val2017'),
    output_dir=Path('data/person_dataset'),
    max_images=2000  # Limitar para demo rápida
)
print(f"Dataset: {stats}")

In [ ]:
from src.dataset.splitter import split_dataset

# Split train/val/test
split_stats = split_dataset(
    source_dir=Path('data/person_dataset'),
    output_dir=Path('data/person_yolo'),
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    seed=42
)
print(f"Split: {split_stats}")

# Crear dataset.yaml
yaml_path = create_dataset_yaml(Path('data/person_yolo'))
print(f"Dataset config: {yaml_path}")

In [ ]:
from src.dataset.augmentation import augment_dataset

# Aumentar datos de entrenamiento
aug_stats = augment_dataset(
    source_dir=Path('data/person_yolo/train'),
    output_dir=Path('data/person_yolo/train'),
    augmentations_per_image=2
)
print(f"Augmentation: {aug_stats}")

## 2. Entrenamiento — Transfer Learning

Entrenamos 4 variantes con pesos preentrenados en COCO:

| Modelo | Licencia (código / pesos) | Parámetros |
|--------|---------------------------|------------|
| YOLOv8n | AGPL-3.0 / AGPL-3.0 | 3.2M |
| YOLOv8s | AGPL-3.0 / AGPL-3.0 | 11.2M |
| YOLOv8m | AGPL-3.0 / AGPL-3.0 | 25.9M |
| YOLO-NAS-S | Apache-2.0 / LICENSE.YOLONAS (no comercial) | ~12M |

In [ ]:
from src.training.trainer import run_experiment
from src.training.config import VARIANT_CONFIGS

dataset_yaml = str(Path('data/person_yolo/dataset.yaml'))
results = {}

for variant_name in VARIANT_CONFIGS.keys():
    print(f"\n{'='*60}")
    print(f"Entrenando: {variant_name}")
    print(f"{'='*60}")
    
    record = run_experiment(
        variant=variant_name,
        dataset_yaml=dataset_yaml,
        device='cuda',
    )
    results[variant_name] = record
    print(f"Completado: {variant_name} — mAP50: {record.metrics.get('mAP50', 'N/A')}")

## 3. Evaluación y comparación

In [ ]:
from src.evaluation.compare import build_comparison_table, format_comparison_markdown

# Cargar todos los resultados
table = build_comparison_table('experiments')
print(format_comparison_markdown(table))

In [ ]:
from src.evaluation.benchmark import benchmark_model
import json

# Benchmark de latencia
for exp_file in sorted(Path('experiments').glob('exp_*.json')):
    data = json.loads(exp_file.read_text())
    variant = data.get('model_variant', '')
    weights = data.get('weights_path', '')
    if weights and Path(weights).exists():
        bench = benchmark_model(weights, device='cuda', img_size=640)
        print(f"{variant}: {bench['mean_latency_ms']:.1f}ms, {bench['fps']:.1f} FPS")

In [ ]:
# Visualizar comparación
import matplotlib.pyplot as plt

if table:
    names = [r['model'] for r in table]
    map50_vals = [float(r['mAP50']) if r['mAP50'] != '-' else 0 for r in table]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444'][:len(names)]
    ax.barh(names, map50_vals, color=colors)
    ax.set_xlabel('mAP@50')
    ax.set_title('Precisión por modelo (mAP@50)')
    
    plt.tight_layout()
    plt.savefig('comparison.png', dpi=150)
    plt.show()
    print("Gráfico guardado en comparison.png")

## 4. Selección y exportación del mejor modelo

Criterios de selección:
- **Licencia**: YOLOv8 es AGPL-3.0; los pesos de YOLO-NAS prohíben uso comercial. Para SaaS de código cerrado la alternativa permisiva evaluada es RF-DETR (Apache-2.0).
- **Precisión**: mAP@50 > 0.80 es aceptable para vigilancia
- **Velocidad**: > 15 FPS requerido para procesamiento en tiempo real

In [ ]:
import shutil

# Seleccionar el mejor modelo (ajustar según resultados reales)
best_variant = 'yolov8s'  # Cambiar según los resultados
best_model_path = Path(f'runs/train/{best_variant}_person/weights/best.pt')

if best_model_path.exists():
    export_dir = Path('exported_models')
    export_dir.mkdir(exist_ok=True)
    shutil.copy2(best_model_path, export_dir / f'saferest360_{best_variant}_v1.pt')
    print(f"Modelo exportado: exported_models/saferest360_{best_variant}_v1.pt")
else:
    print(f"Modelo no encontrado: {best_model_path}")
    print("Asegúrate de ejecutar las celdas de entrenamiento primero.")

In [ ]:
# Respaldar resultados (el disco de Colab se borra al desconectarse el runtime)
import zipfile

backup = Path('saferest360_resultados.zip')
patterns = ['experiments/*.json', 'runs/train/*/results.csv', 'runs/train/*/args.yaml',
            'runs/train/*/weights/best.pt', 'comparison.png', 'exported_models/*.pt']
with zipfile.ZipFile(backup, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in patterns:
        for f in Path('.').glob(pattern):
            zf.write(f)
            print(f"  + {f}")
print(f"Respaldo: {backup} ({backup.stat().st_size / 1e6:.1f} MB)")

from google.colab import files
files.download(str(backup))